In [4]:
import ultralytics
ultralytics.checks()

Ultralytics 8.4.171  Python-3.11.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Setup complete  (16 CPUs, 31.2 GB RAM, 413.8/475.5 GB disk)


In [5]:
! nvidia-smi

Wed Oct  7 13:26:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.88                 KMD Version: 610.88        CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   48C    P8              2W /   81W |     415MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Import Data

In [6]:
# Import necessary libraries
import os
import requests
import tarfile
from tqdm import tqdm
     

In [7]:
yaml_content = """
  path: ./dataset
  train: images/train
  val: images/val
  nc: 8
  names: ["Car", "Motorcycle", "Bus", "Truck", "Tuktuk", "Van", "Pickup", "Songthaew"]
  """
with open("data.yaml", "w") as f:
      f.write(yaml_content)

## Step 2: Data Preprocessing โ€” Convert `train.csv` to YOLO format

`train.csv` stores one row per bounding box in pixel corner coordinates
(`x1, y1, x2, y2`). YOLO (Ultralytics) instead expects, per image, a `.txt`
label file with one line per box:

```
class_id x_center y_center width height
```

all normalized to `[0, 1]` by image width/height, with images and labels
laid out as:

```
dataset/
  images/train/*.jpg
  images/val/*.jpg
  labels/train/*.txt
  labels/val/*.txt
```

which matches the `path`/`train`/`val` keys already written to `data.yaml`.

**Train/val split strategy:** the real test set (`test/test/`) comes from 5
camera locations that do **not** appear anywhere in `train/train/` (15
different cameras). To get a validation score that actually estimates
cross-camera generalization (instead of an optimistic same-camera score),
we hold out a few whole **cameras** from training rather than splitting
images randomly.

In [8]:
# --- Imports ---
import random
import shutil
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from tqdm import tqdm


In [9]:
# --- Paths & config ---
BASE_DIR = Path(".")
TRAIN_IMG_DIR = BASE_DIR / "train" / "train"
TEST_IMG_DIR = BASE_DIR / "test" / "test"
TRAIN_CSV = BASE_DIR / "train.csv"

DATASET_DIR = BASE_DIR / "dataset"
IMG_TRAIN_OUT = DATASET_DIR / "images" / "train"
IMG_VAL_OUT = DATASET_DIR / "images" / "val"
LBL_TRAIN_OUT = DATASET_DIR / "labels" / "train"
LBL_VAL_OUT = DATASET_DIR / "labels" / "val"

CLASS_NAMES = ["Car", "Motorcycle", "Bus", "Truck", "Tuktuk", "Van", "Pickup", "Songthaew"]

# Hold out these camera IDs for validation so val measures generalization to
# UNSEEN cameras -- matching how the real test set (different cameras) is scored.
# ~3 of the 15 train cameras (~500-600 images, roughly 15-20% of train).
VAL_CAMERA_IDS = {"172", "229", "1437"}

RANDOM_SEED = 42
random.seed(RANDOM_SEED)


In [10]:
# --- Load ground-truth boxes & derive camera id per image ---
df_train = pd.read_csv(TRAIN_CSV)
df_train["camera_id"] = df_train["image_id"].str.extract(r"^(\d+)_")

all_cameras = sorted(df_train["camera_id"].unique(), key=int)
print(f"Train cameras ({len(all_cameras)}):", all_cameras)
print("Held-out (val) cameras:", sorted(VAL_CAMERA_IDS, key=int))

missing = VAL_CAMERA_IDS - set(all_cameras)
assert not missing, f"VAL_CAMERA_IDS not found in train data: {missing}"


Train cameras (15): ['12', '172', '180', '182', '214', '222', '229', '231', '232', '244', '1066', '1407', '1426', '1427', '1437']
Held-out (val) cameras: ['172', '229', '1437']


In [11]:
# --- Split image ids by camera (not randomly by image) ---
all_image_ids = df_train["image_id"].unique().tolist()
image_camera = df_train.drop_duplicates("image_id").set_index("image_id")["camera_id"]

val_image_ids = [img for img in all_image_ids if image_camera[img] in VAL_CAMERA_IDS]
train_image_ids = [img for img in all_image_ids if image_camera[img] not in VAL_CAMERA_IDS]

n_train_boxes = df_train[df_train["image_id"].isin(train_image_ids)].shape[0]
n_val_boxes = df_train[df_train["image_id"].isin(val_image_ids)].shape[0]

print(f"Train images: {len(train_image_ids)}  |  Val images: {len(val_image_ids)}")
print(f"Train boxes:  {n_train_boxes}  |  Val boxes:  {n_val_boxes}")


Train images: 2297  |  Val images: 592
Train boxes:  23724  |  Val boxes:  3672


In [12]:
def to_yolo_format(x1, y1, x2, y2, img_w, img_h):
    """Convert pixel corner box (x1, y1, x2, y2) to normalized YOLO
    (x_center, y_center, width, height) in [0, 1]."""
    x_center = (x1 + x2) / 2.0 / img_w
    y_center = (y1 + y2) / 2.0 / img_h
    width = (x2 - x1) / img_w
    height = (y2 - y1) / img_h
    return (
        min(max(x_center, 0.0), 1.0),
        min(max(y_center, 0.0), 1.0),
        min(max(width, 0.0), 1.0),
        min(max(height, 0.0), 1.0),
    )


In [13]:
# --- Create output folder structure expected by data.yaml ---
for d in (IMG_TRAIN_OUT, IMG_VAL_OUT, LBL_TRAIN_OUT, LBL_VAL_OUT):
    d.mkdir(parents=True, exist_ok=True)


In [14]:
def build_split(image_ids, img_out_dir, lbl_out_dir, src_img_dir, df):
    """Copy images into img_out_dir and write one YOLO label .txt per
    image into lbl_out_dir (empty file if the image has no boxes)."""
    boxes_by_image = df.groupby("image_id")

    for image_id in tqdm(image_ids, desc=f"Processing {img_out_dir.name}"):
        src_path = src_img_dir / image_id
        dst_img_path = img_out_dir / image_id
        if not dst_img_path.exists():
            shutil.copy2(src_path, dst_img_path)

        with Image.open(src_path) as im:
            img_w, img_h = im.size

        label_lines = []
        if image_id in boxes_by_image.groups:
            for _, row in boxes_by_image.get_group(image_id).iterrows():
                xc, yc, w, h = to_yolo_format(row.x1, row.y1, row.x2, row.y2, img_w, img_h)
                label_lines.append(f"{int(row.class_id)} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

        label_path = lbl_out_dir / (Path(image_id).stem + ".txt")
        label_path.write_text("\n".join(label_lines), encoding="utf-8")


build_split(train_image_ids, IMG_TRAIN_OUT, LBL_TRAIN_OUT, TRAIN_IMG_DIR, df_train)
build_split(val_image_ids, IMG_VAL_OUT, LBL_VAL_OUT, TRAIN_IMG_DIR, df_train)

print("Done converting train.csv -> YOLO labels.")


Processing train:   0%|          | 0/2297 [00:00<?, ?it/s]

Processing val: 100%|██████████| 592/592 [00:00<00:00, 804.11it/s]

Done converting train.csv -> YOLO labels.


In [15]:
# --- Sanity check: counts + visualize a sample image with its converted boxes ---
n_train_imgs = len(list(IMG_TRAIN_OUT.iterdir()))
n_val_imgs = len(list(IMG_VAL_OUT.iterdir()))
n_train_lbls = len(list(LBL_TRAIN_OUT.iterdir()))
n_val_lbls = len(list(LBL_VAL_OUT.iterdir()))
print(f"images/train: {n_train_imgs}  labels/train: {n_train_lbls}")
print(f"images/val:   {n_val_imgs}  labels/val:   {n_val_lbls}")
assert n_train_imgs == n_train_lbls
assert n_val_imgs == n_val_lbls


def show_sample(image_id, img_dir, lbl_dir, class_names):
    img_path = img_dir / image_id
    lbl_path = lbl_dir / (Path(image_id).stem + ".txt")

    im = Image.open(img_path)
    img_w, img_h = im.size

    fig, ax = plt.subplots(1, figsize=(6, 5))
    ax.imshow(im)

    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            if not line.strip():
                continue
            cls, xc, yc, w, h = line.split()
            cls = int(cls)
            xc, yc = float(xc) * img_w, float(yc) * img_h
            w, h = float(w) * img_w, float(h) * img_h
            x1, y1 = xc - w / 2, yc - h / 2
            rect = patches.Rectangle((x1, y1), w, h, linewidth=1.5, edgecolor="red", facecolor="none")
            ax.add_patch(rect)
            ax.text(x1, max(y1 - 2, 0), class_names[cls], color="red", fontsize=8)

    ax.set_title(image_id)
    ax.axis("off")
    plt.show()


show_sample(train_image_ids[0], IMG_TRAIN_OUT, LBL_TRAIN_OUT, CLASS_NAMES)
show_sample(val_image_ids[0], IMG_VAL_OUT, LBL_VAL_OUT, CLASS_NAMES)


images/train: 11324  labels/train: 5898
images/val:   1208  labels/val:   604


AssertionError: 

### Standalone reproducible preprocessing cell

Run the next cell by itself from the `2110531_mid_term` directory. It reads `train.csv`, converts the annotations to YOLO format, copies the images, and writes `dataset/data.yaml`.


In [ ]:
# ============================================================
# Reproducible preprocessing: train.csv -> Ultralytics YOLO format
# Run this single cell from the 2110531_mid_term directory.
# ============================================================

from pathlib import Path
import re
import shutil

import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

# 1. Input and output paths
BASE_DIR = Path.cwd()
TRAIN_IMAGE_DIR = BASE_DIR / "train" / "train"
ANNOTATION_CSV = BASE_DIR / "train.csv"
YOLO_DIR = BASE_DIR / "dataset"

# Hold out complete cameras for validation. The test set uses different cameras,
# so this is a more realistic validation split than a random image split.
VAL_CAMERA_IDS = {"172", "229", "1437"}

CLASS_NAMES = [
    "Car", "Motorcycle", "Bus", "Truck",
    "Tuktuk", "Van", "Pickup", "Songthaew",
]

# Check that the notebook is being run from the correct folder.
assert TRAIN_IMAGE_DIR.exists(), f"Missing folder: {TRAIN_IMAGE_DIR}"
assert ANNOTATION_CSV.exists(), f"Missing file: {ANNOTATION_CSV}"

# 2. Read annotations and identify camera IDs
annotations = pd.read_csv(ANNOTATION_CSV)
required_columns = {"image_id", "class_id", "x1", "y1", "x2", "y2"}
missing_columns = required_columns - set(annotations.columns)
assert not missing_columns, f"Missing CSV columns: {missing_columns}"

annotations["camera_id"] = annotations["image_id"].str.extract(r"^(\d+)_")[0]

# 3. Use every train image, including images with zero annotations
image_paths = sorted(TRAIN_IMAGE_DIR.glob("*.jpg"))
assert image_paths, f"No .jpg images found in {TRAIN_IMAGE_DIR}"

image_ids = [path.name for path in image_paths]
image_camera = {
    image_id: re.match(r"^(\d+)_", image_id).group(1)
    for image_id in image_ids
}

missing_val_cameras = VAL_CAMERA_IDS - set(image_camera.values())
assert not missing_val_cameras, (
    f"Validation camera IDs not found in train images: {missing_val_cameras}"
)

train_image_ids = [
    image_id for image_id in image_ids
    if image_camera[image_id] not in VAL_CAMERA_IDS
]
val_image_ids = [
    image_id for image_id in image_ids
    if image_camera[image_id] in VAL_CAMERA_IDS
]

# 4. Prepare the Ultralytics folder structure
split_dirs = {
    "train": {
        "images": YOLO_DIR / "images" / "train",
        "labels": YOLO_DIR / "labels" / "train",
        "image_ids": train_image_ids,
    },
    "val": {
        "images": YOLO_DIR / "images" / "val",
        "labels": YOLO_DIR / "labels" / "val",
        "image_ids": val_image_ids,
    },
}

for split in split_dirs.values():
    split["images"].mkdir(parents=True, exist_ok=True)
    split["labels"].mkdir(parents=True, exist_ok=True)

# Group once so each image's boxes can be accessed efficiently.
boxes_by_image = annotations.groupby("image_id")

# 5. Convert pixel boxes to normalized YOLO boxes and write label files
# YOLO line format:
# class_id x_center y_center box_width box_height
for split_name, split in split_dirs.items():
    for image_id in tqdm(split["image_ids"], desc=f"Preparing {split_name}"):
        source_image = TRAIN_IMAGE_DIR / image_id
        destination_image = split["images"] / image_id
        shutil.copy2(source_image, destination_image)

        with Image.open(source_image) as image:
            image_width, image_height = image.size

        yolo_lines = []
        if image_id in boxes_by_image.groups:
            image_boxes = boxes_by_image.get_group(image_id)
            for row in image_boxes.itertuples(index=False):
                # Clip boxes to image boundaries before normalization.
                x1 = max(0.0, min(float(row.x1), image_width))
                y1 = max(0.0, min(float(row.y1), image_height))
                x2 = max(0.0, min(float(row.x2), image_width))
                y2 = max(0.0, min(float(row.y2), image_height))

                box_width = x2 - x1
                box_height = y2 - y1
                if box_width <= 0 or box_height <= 0:
                    continue

                x_center = ((x1 + x2) / 2.0) / image_width
                y_center = ((y1 + y2) / 2.0) / image_height
                normalized_width = box_width / image_width
                normalized_height = box_height / image_height

                yolo_lines.append(
                    f"{int(row.class_id)} "
                    f"{x_center:.6f} {y_center:.6f} "
                    f"{normalized_width:.6f} {normalized_height:.6f}"
                )

        label_path = split["labels"] / f"{Path(image_id).stem}.txt"
        label_path.write_text("\n".join(yolo_lines), encoding="utf-8")

# 6. Write/refresh the dataset configuration used by Ultralytics
DATA_YAML = YOLO_DIR / "data.yaml"
DATA_YAML.write_text(
    "\n".join([
        f"path: {YOLO_DIR.resolve().as_posix()}",
        "train: images/train",
        "val: images/val",
        f"nc: {len(CLASS_NAMES)}",
        f"names: {CLASS_NAMES}",
        "",
    ]),
    encoding="utf-8",
)

# 7. Verify that each copied image has a corresponding label file
for split_name, split in split_dirs.items():
    image_count = len(list(split["images"].glob("*.jpg")))
    label_count = len(list(split["labels"].glob("*.txt")))
    expected_count = len(split["image_ids"])
    assert image_count == expected_count, (split_name, image_count, expected_count)
    assert label_count == expected_count, (split_name, label_count, expected_count)

print("Preprocessing complete.")
print(f"Train images: {len(train_image_ids)}")
print(f"Validation images: {len(val_image_ids)}")
print(f"YOLO dataset config: {DATA_YAML}")
print(f"YOLO dataset directory: {YOLO_DIR.resolve()}")


Preparing train:   0%|          | 0/2387 [00:00<?, ?it/s]

Preparing val:   0%|          | 0/604 [00:00<?, ?it/s]

Preprocessing complete.
Train images: 2387
Validation images: 604
YOLO dataset config: C:\Users\Krisda Taveesak\Documents\python_vsc\master_project\2110531_DSDE_2026s1\2110531_mid_term\dataset\data.yaml
YOLO dataset directory: C:\Users\Krisda Taveesak\Documents\python_vsc\master_project\2110531_DSDE_2026s1\2110531_mid_term\dataset


In [ ]:
# ============================================================
# Oversample rare-class images to fix class imbalance
# Copies images + labels N times so rare classes appear more often
# Run ONCE after build_split; re-running is safe (skips existing copies)
# ============================================================
import shutil
from pathlib import Path
import pandas as pd

# --- Config -------------------------------------------------------------
DATASET_DIR   = Path("dataset")
IMG_TRAIN_DIR = DATASET_DIR / "images" / "train"
LBL_TRAIN_DIR = DATASET_DIR / "labels" / "train"
TRAIN_CSV     = Path("train.csv")

CLASS_NAMES = ["Car", "Motorcycle", "Bus", "Truck",
               "Tuktuk", "Van", "Pickup", "Songthaew"]

# How many EXTRA copies to make for each rare class
# (images that contain at least one instance of that class get duplicated)
# Reduced multipliers to cut dataset size ~50% and speed up training
OVERSAMPLE_MULTIPLIER = {
    7: 4,   # Songthaew  117 boxes  -> ~5x
    4: 3,   # Tuktuk     265 boxes  -> ~4x
    5: 2,   # Van        354 boxes  -> ~3x
    2: 2,   # Bus        482 boxes  -> ~3x
    6: 1,   # Pickup     497 boxes  -> ~2x
    3: 1,   # Truck     1356 boxes  -> ~2x
}  # Car (0) and Motorcycle (1) are NOT oversampled

# --- Remove old oversampled copies before re-running --------------------
old_copies = list(IMG_TRAIN_DIR.glob("*_os*_*.jpg"))
for f in old_copies:
    f.unlink()
    lbl = LBL_TRAIN_DIR / (f.stem + ".txt")
    if lbl.exists():
        lbl.unlink()
print(f"Removed {len(old_copies)} old oversampled copies")

# --- Load train split image list from the labels folder -----------------
df = pd.read_csv(TRAIN_CSV)

# Only look at images that are already in dataset/images/train
existing_images = {p.name for p in IMG_TRAIN_DIR.glob("*.jpg")}
df_train = df[df["image_id"].isin(existing_images)]

# Build a map: class_id -> list of image_ids that contain it
class_to_images = (
    df_train.groupby("class_id")["image_id"]
    .apply(lambda x: list(x.unique()))
    .to_dict()
)

# --- Oversample ----------------------------------------------------------
copied = 0
skipped = 0

for class_id, extra_copies in OVERSAMPLE_MULTIPLIER.items():
    images_for_class = class_to_images.get(class_id, [])
    class_name = CLASS_NAMES[class_id]
    print(f"Class {class_id} ({class_name}): {len(images_for_class)} images, "
          f"making {extra_copies} extra copies each")

    for img_name in images_for_class:
        stem = Path(img_name).stem   # filename without .jpg
        src_img = IMG_TRAIN_DIR / img_name
        src_lbl = LBL_TRAIN_DIR / (stem + ".txt")

        if not src_img.exists() or not src_lbl.exists():
            continue

        for copy_idx in range(1, extra_copies + 1):
            dst_img = IMG_TRAIN_DIR / f"{stem}_os{class_id}_{copy_idx}.jpg"
            dst_lbl = LBL_TRAIN_DIR / f"{stem}_os{class_id}_{copy_idx}.txt"

            if dst_img.exists():
                skipped += 1
                continue

            shutil.copy2(src_img, dst_img)
            shutil.copy2(src_lbl, dst_lbl)
            copied += 1

print(f"\nDone. {copied} new image+label pairs added, {skipped} already existed.")

Removed 4497 old oversampled copies
Class 7 (Songthaew): 58 images, making 4 extra copies each
Class 4 (Tuktuk): 150 images, making 3 extra copies each
Class 5 (Van): 234 images, making 2 extra copies each
Class 2 (Bus): 326 images, making 2 extra copies each
Class 6 (Pickup): 386 images, making 1 extra copies each
Class 3 (Truck): 851 images, making 1 extra copies each

Done. 3039 new image+label pairs added, 0 already existed.


In [27]:
# ============================================================
# Drop train images where Car(0) + Motorcycle(1) > 90% of the boxes
# Source: dataset/  ->  Output: dataset_balanced/
# Val is NOT copied or changed: it stays the full 8-class val set.
# ============================================================
import os, shutil
from collections import Counter
from pathlib import Path

SRC_DIR = Path("dataset")
DST_DIR = Path("dataset_balanced")

DOMINANT = {0, 1}              # Car, Motorcycle
DROP_THRESHOLD = 0.70          # drop image if (class0 + class1) / total boxes > this the less value, more picture get drop
KEEP_EMPTY_IMAGES = True       # images with no boxes (pure background): keep or drop
CLASS_NAMES = ["Car", "Motorcycle", "Bus", "Truck", "Tuktuk", "Van", "Pickup", "Songthaew"]

if DST_DIR.exists():
    shutil.rmtree(DST_DIR)

src_img, src_lbl = SRC_DIR / "images" / "train", SRC_DIR / "labels" / "train"
dst_img, dst_lbl = DST_DIR / "images" / "train", DST_DIR / "labels" / "train"
dst_img.mkdir(parents=True)
dst_lbl.mkdir(parents=True)

def link_or_copy(src, dst):
    try:
        os.link(src, dst)          # hard link: no extra disk space
    except OSError:
        shutil.copy2(src, dst)

before, after = Counter(), Counter()
kept = dropped = empty = 0

for img in sorted(src_img.glob("*.jpg")):
    lbl = src_lbl / (img.stem + ".txt")
    ids = [int(l.split()[0]) for l in lbl.read_text().splitlines() if l.strip()] if lbl.exists() else []
    before.update(ids)

    if not ids:
        empty += 1
        keep = KEEP_EMPTY_IMAGES
    else:
        dominant_share = sum(i in DOMINANT for i in ids) / len(ids)
        keep = dominant_share <= DROP_THRESHOLD

    if keep:
        link_or_copy(img, dst_img / img.name)
        link_or_copy(lbl, dst_lbl / lbl.name)
        after.update(ids)
        kept += 1
    else:
        dropped += 1

print(f"Images kept: {kept} | dropped: {dropped} | (empty images seen: {empty})")
print(f"\n{'class':<12}{'before':>8}{'after':>8}")
for i, n in enumerate(CLASS_NAMES):
    print(f"{n:<12}{before[i]:>8}{after[i]:>8}")

(DST_DIR / "data.yaml").write_text(
    "\n".join([
        f"path: {DST_DIR.resolve().as_posix()}",
        "train: images/train",
        f"val: {(SRC_DIR / 'images' / 'val').resolve().as_posix()}",   # original full val set
        f"nc: {len(CLASS_NAMES)}",
        f"names: {CLASS_NAMES}",
        "",
    ]),
    encoding="utf-8",
)
print("\nWrote", DST_DIR / "data.yaml")

Images kept: 1056 | dropped: 4842 | (empty images seen: 90)

class         before   after
Car            46601    2233
Motorcycle     15313    1031
Bus             1917     302
Truck           4176     955
Tuktuk          1106     132
Van             1503     236
Pickup          1609     457
Songthaew        667     561

Wrote dataset_balanced\data.yaml


In [29]:
import pandas as pd

file_path_train = "./train.csv"
df_train = pd.read_csv(file_path_train, header=0)

CLASS_NAMES = ["Car", "Motorcycle", "Bus", "Truck", "Tuktuk", "Van", "Pickup", "Songthaew"]
DROP_THRESHOLD = 0.50 
DOMINANT = [0, 1]            # Car, Motorcycle      # drop image if (class0 + class1) / total boxes > this
VAL_CAMERA_IDS = {"172", "229", "1437"}

# keep only the TRAIN cameras (same split as the dataset/ folder)
df_train["camera_id"] = df_train["image_id"].str.extract(r"^(\d+)_")[0]
df_train = df_train[~df_train["camera_id"].isin(VAL_CAMERA_IDS)]

# share of Car + Motorcycle boxes in each image
dominant_share = (
    df_train["class_id"].isin(DOMINANT)
    .groupby(df_train["image_id"]).mean()
)
drop_ids = dominant_share[dominant_share > DROP_THRESHOLD].index
df_kept = df_train[~df_train["image_id"].isin(drop_ids)]

print(f"images before: {df_train['image_id'].nunique()}")
print(f"images dropped: {len(drop_ids)}")
print(f"images after:  {df_kept['image_id'].nunique()}")


# ---- 1) class distribution: before vs after ----
dist = pd.DataFrame({
    "before":        df_train["class_id"].value_counts().sort_index(),
    "after":         df_kept["class_id"].value_counts().sort_index(),
    "before_share":  df_train["class_id"].value_counts(normalize=True).sort_index(),
    "after_share":   df_kept["class_id"].value_counts(normalize=True).sort_index(),
}).fillna(0)
dist["retained_%"] = dist["after"] / dist["before"] * 100
dist.index = [CLASS_NAMES[i] for i in dist.index]
print("\n=== class distribution (boxes) ===")
print(dist.round(3))


# ---- 2) objects per image (all classes): before vs after ----
print("\n=== objects per image: before ===")
print(df_train.groupby("image_id").size().describe())
print("\n=== objects per image: after ===")
print(df_kept.groupby("image_id").size().describe())


# ---- 3) avg + sd per class per image: before vs after ----
def per_class_report(df):
    return (
        df.groupby(["image_id", "class_id"]).size()
          .unstack("class_id", fill_value=0)    # one column per class, zeros filled
          .reindex(columns=range(len(CLASS_NAMES)), fill_value=0)
          .agg(["mean", "std"])                 # avg & sd per class across images
          .T                                    # rows = class_id, cols = mean, std
    )

report = per_class_report(df_train).join(
    per_class_report(df_kept), lsuffix="_before", rsuffix="_after"
)
report.index = CLASS_NAMES
print("\n=== avg / sd of boxes per image, per class ===")
print(report.round(3))

images before: 2297
images dropped: 2025
images after:  272

=== class distribution (boxes) ===
            before  after  before_share  after_share  retained_%
Car          15533    539         0.655        0.371       3.470
Motorcycle    5512    262         0.232        0.180       4.753
Bus            387     63         0.016        0.043      16.279
Truck         1226    287         0.052        0.198      23.409
Tuktuk         185     22         0.008        0.015      11.892
Van            290     51         0.012        0.035      17.586
Pickup         475    127         0.020        0.087      26.737
Songthaew      116    101         0.005        0.070      87.069

=== objects per image: before ===
count    2297.000000
mean       10.328254
std         7.227587
min         1.000000
25%         5.000000
50%         9.000000
75%        14.000000
max        48.000000
dtype: float64

=== objects per image: after ===
count    272.000000
mean       5.338235
std        3.602419
min    

In [28]:
# ============================================================
# CHECK: does dataset_balanced still contain old / wrong pictures?
# Read-only. Re-applies the drop rule to dataset/ and compares.
# ============================================================
from pathlib import Path

SRC_DIR = Path("dataset")
DST_DIR = Path("dataset_balanced")

DOMINANT = {0, 1}
DROP_THRESHOLD = 0.70          # must match the value used to build the folder
KEEP_EMPTY_IMAGES = True

src_img, src_lbl = SRC_DIR / "images" / "train", SRC_DIR / "labels" / "train"
dst_img, dst_lbl = DST_DIR / "images" / "train", DST_DIR / "labels" / "train"

def read_ids(p):
    return [int(l.split()[0]) for l in p.read_text().splitlines() if l.strip()] if p.exists() else []

# 1) What SHOULD be in dataset_balanced according to the rule
expected = set()
for img in src_img.glob("*.jpg"):
    ids = read_ids(src_lbl / (img.stem + ".txt"))
    keep = KEEP_EMPTY_IMAGES if not ids else sum(i in DOMINANT for i in ids) / len(ids) <= DROP_THRESHOLD
    if keep:
        expected.add(img.name)

# 2) What IS in dataset_balanced
actual = {p.name for p in dst_img.glob("*.jpg")}

stale          = actual - expected                                   # should have been dropped / deleted
missing        = expected - actual                                   # should be there but isn't
not_in_source  = {n for n in actual if not (src_img / n).exists()}   # image no longer exists in dataset/
label_differs  = [n for n in actual & expected
                  if read_ids(dst_lbl / (Path(n).stem + ".txt")) != read_ids(src_lbl / (Path(n).stem + ".txt"))]
size_differs   = [n for n in actual & expected
                  if (dst_img / n).stat().st_size != (src_img / n).stat().st_size]
orphan_labels  = {p.stem for p in dst_lbl.glob("*.txt")} - {Path(n).stem for n in actual}

def share(n):
    ids = read_ids(dst_lbl / (Path(n).stem + ".txt"))
    return sum(i in DOMINANT for i in ids) / len(ids) if ids else 0.0
over_threshold = [n for n in actual if share(n) > DROP_THRESHOLD]

print(f"expected by rule: {len(expected)} | actually in dataset_balanced: {len(actual)}")
print(f"stale (should NOT be there):        {len(stale)}")
print(f"missing (should be there):          {len(missing)}")
print(f"image not in source dataset/ any more: {len(not_in_source)}")
print(f"label differs from source:          {len(label_differs)}")
print(f"image file differs from source:     {len(size_differs)}")
print(f"orphan labels (no image):           {len(orphan_labels)}")
print(f"kept images above threshold:        {len(over_threshold)}")
print(f"_os copies  dataset/: {len(list(src_img.glob('*_os*_*.jpg')))} | "
      f"dataset_balanced/: {len(list(dst_img.glob('*_os*_*.jpg')))}")
for name, items in [("stale", stale), ("not_in_source", not_in_source), ("over_threshold", over_threshold)]:
    if items:
        print(f"  e.g. {name}:", sorted(items)[:5])

old_retained = bool(stale or not_in_source or label_differs or size_differs or orphan_labels or over_threshold)
print("\nOLD PICTURES RETAINED:", old_retained)
if (DST_DIR / "labels" / "train.cache").exists():
    print("note: labels/train.cache exists (Ultralytics writes it during training; delete it after rebuilding)")

expected by rule: 1056 | actually in dataset_balanced: 1056
stale (should NOT be there):        0
missing (should be there):          0
image not in source dataset/ any more: 0
label differs from source:          0
image file differs from source:     0
orphan labels (no image):           0
kept images above threshold:        0
_os copies  dataset/: 3039 | dataset_balanced/: 660

OLD PICTURES RETAINED: False


In [26]:
from ultralytics import RTDETR

MODEL_NAME = "rtdetr-l.pt"  # or "rtdetr-x.pt" for higher capacity

BATCH_SIZE = {
    "rtdetr-l.pt": 4,
    "rtdetr-x.pt": 2,
}[MODEL_NAME]

model = RTDETR(MODEL_NAME)

results = model.train(
    data="dataset_balanced/data.yaml",   # <- balanced train set (threshold rule), full 8-class val
    epochs=50,
    patience=10,
    batch=BATCH_SIZE,
    device=0,
    workers=2,
    pretrained=True,
    plots=True,
    amp=True,
    seed=0,

    # Resolution: tiny objects, so upscale more than 640
    imgsz=800,            # drop back to 640 if you run out of GPU memory

    # Optimizer
    optimizer="AdamW",
    lr0=1e-4,
    lrf=0.05,
    weight_decay=1e-4,
    warmup_epochs=2,
    cos_lr=True,

    # Augmentation for fixed-camera traffic scenes
    mosaic=0.5,
    close_mosaic=10,
    scale=0.5,            # default 0.5 shrinks tiny vehicles until they vanish
    fliplr=0.5,
    flipud=0.0,
    degrees=0.0,
    mixup=0.0,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.5,            # day, night and glare changes

    project="runs/detect",
    name=MODEL_NAME.replace(".pt", "") + "_balanced",   # runs/detect/rtdetr-l_balanced
    exist_ok=True,
)

New https://pypi.org/project/ultralytics/8.4.174 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.171  Python-3.11.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=dataset_balanced/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.5, hsv_v=0.5, imgsz=800, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.05, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rt

C:\Users\Krisda Taveesak\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\autograd\graph.py:823: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:96.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50      4.83G      2.095      1.769     0.8263         44        800: 3% ──────────── 19/518 2.5it/s 16.0s<3:17

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x00000266359C2950>>
Traceback (most recent call last):
  File "C:\Users\Krisda Taveesak\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\ipykernel\ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


       1/50      4.88G      1.767     0.9269     0.6261         41        800: 11% ━─────────── 61/518 2.6it/s 32.4s<2:54


KeyboardInterrupt: 

## Step 4 (Submission): Run inference on test images and build `submission.csv`

After training, load the best checkpoint and run it over every image in `test/test/`.
The output is a CSV with columns `id, image_id, class_id, confidence, x1, y1, x2, y2`.

**How `image_id` is built:** The `image_id` is the **short filename** of each test image
(e.g. `227_20260825_101120.jpg`), which is the same format used in `train.csv`.
This matches the actual filenames in `test/test/` directly — no mapping needed.

Adjust `CONF_THRESHOLD` and `IOU_THRESHOLD` before your final submission.

In [ ]:
from pathlib import Path
import pandas as pd
from PIL import Image
from ultralytics import RTDETR
from tqdm import tqdm

# -- Config ------------------------------------------------------------------
MODEL_WEIGHTS  = "runs/weights/best.pt"  # path to trained checkpoint
TEST_IMG_DIR   = Path("test/test")        # folder with test images
OUTPUT_CSV     = Path("submission.csv")   # file to submit

CONF_THRESHOLD = 0.25   # keep boxes with confidence >= this value
IOU_THRESHOLD  = 0.45   # NMS IoU threshold
IMG_SIZE       = 640    # must match training imgsz

# -- Load model --------------------------------------------------------------
model = RTDETR(MODEL_WEIGHTS)

# -- Collect test images -----------------------------------------------------
test_images = sorted(TEST_IMG_DIR.glob("*.jpg"))
print(f"Found {len(test_images)} test images")

# -- Run inference & collect rows --------------------------------------------
# image_id is the short filename (e.g. 227_20260825_101120.jpg),
# which is the same format as train.csv uses.
rows = []
row_id = 0

for img_path in tqdm(test_images, desc="Inferring"):
    image_id = img_path.name  # e.g. 227_20260825_101120.jpg

    # Get image dimensions for clamping
    with Image.open(img_path) as im:
        img_w, img_h = im.size

    results = model.predict(
        source  = str(img_path),
        imgsz   = IMG_SIZE,
        conf    = CONF_THRESHOLD,
        iou     = IOU_THRESHOLD,
        device  = 0,           # change to "cpu" if no GPU
        verbose = False,
    )

    result = results[0]
    boxes  = result.boxes

    if boxes is not None and len(boxes) > 0:
        xyxy    = boxes.xyxy.cpu().numpy()             # shape (N, 4) pixel coords
        confs   = boxes.conf.cpu().numpy()             # shape (N,)
        cls_ids = boxes.cls.cpu().numpy().astype(int)  # shape (N,)

        for i in range(len(xyxy)):
            x1, y1, x2, y2 = xyxy[i]
            # Clamp to valid pixel range
            x1 = max(0.0, float(x1))
            y1 = max(0.0, float(y1))
            x2 = min(float(img_w), float(x2))
            y2 = min(float(img_h), float(y2))
            rows.append({
                "id"        : row_id,
                "image_id"  : image_id,
                "class_id"  : cls_ids[i],
                "confidence": round(float(confs[i]), 6),
                "x1"        : round(x1, 2),
                "y1"        : round(y1, 2),
                "x2"        : round(x2, 2),
                "y2"        : round(y2, 2),
            })
            row_id += 1
    else:
        # No detections: write one dummy row so this image_id appears in the file.
        rows.append({
            "id"        : row_id,
            "image_id"  : image_id,
            "class_id"  : 0,
            "confidence": 0.0,
            "x1": 0, "y1": 0, "x2": 1, "y2": 1,
        })
        row_id += 1

# -- Build DataFrame & save --------------------------------------------------
df_sub = pd.DataFrame(rows, columns=["id","image_id","class_id","confidence","x1","y1","x2","y2"])

# Coordinate sanity-check
assert (df_sub["x1"] >= 0).all(), "x1 has negative values!"
assert (df_sub["y1"] >= 0).all(), "y1 has negative values!"
assert (df_sub["x2"] >= df_sub["x1"]).all(), "x2 < x1 found!"
assert (df_sub["y2"] >= df_sub["y1"]).all(), "y2 < y1 found!"
print("Coordinate sanity check passed.")

df_sub.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df_sub)} rows to {OUTPUT_CSV}")
df_sub.head()


Found 1013 test images


Inferring: 100%|██████████| 1013/1013 [00:37<00:00, 27.17it/s]


Coordinate sanity check passed.
Saved 20206 rows to submission.csv


,id,image_id,class_id,confidence,x1,y1,x2,y2
0,0,1068_20260825_060110.jpg,0,0.942764,129.59,112.00,173.00,161.96
1,1,1068_20260825_060110.jpg,0,0.942148,49.38,168.47,103.28,217.48
2,2,1068_20260825_060110.jpg,0,0.898396,184.87,21.50,199.19,38.68
3,3,1068_20260825_060110.jpg,0,0.896465,270.22,24.77,290.79,45.75
4,4,1068_20260825_060110.jpg,0,0.895268,209.64,32.03,225.28,50.22
